# 02 · ความเชื่อมั่นของแบบวัด DASS-21

DASS-21 ฉบับภาษาไทยผ่านการตรวจคุณภาพกับกลุ่มอื่นมาแล้ว (นักศึกษาพยาบาลไทย นักศึกษามหาวิทยาลัยในเอธิโอเปีย)
แต่ยังไม่เคยตรวจกับกลุ่มตัวอย่างนี้ เค้าโครงจึงกำหนดให้รายงานค่าแอลฟาของครอนบาคจากข้อมูลของเราเองทั้งสามมิติ

- **alpha** พร้อมช่วงความเชื่อมั่น 95% แบบ percentile bootstrap (5,000 รอบ) ค่าตั้งแต่ 0.70 ขึ้นไปถือว่ายอมรับได้
- **item-total r** สหสัมพันธ์ของข้อนั้นกับผลรวมของอีก 6 ข้อในมิติเดียวกัน
- **alpha if deleted** ค่าแอลฟาถ้าตัดข้อนั้นออก ข้อที่อ่อนจะมี r ต่ำ และแอลฟาสูงขึ้นเมื่อตัดออก
- ตารางสุดท้ายคือสหสัมพันธ์ระหว่างคะแนนสามมิติ ถ้าสูงมากแปลว่าสามมิติแยกจากกันไม่ชัดในกลุ่มนี้

In [1]:
import os
import pathlib
import sys

# notebook.nvim starts the kernel in nvim's cwd; step into codingpy/ so common.py
# and data/ are found when the notebook is opened from the repo root
if pathlib.Path("codingpy/common.py").is_file():
    os.chdir("codingpy")
    sys.path.insert(0, os.getcwd())

%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from common import DASS_ITEMS, DIMENSIONS, N_BOOT, SEED, load_data, passed_attention

df = passed_attention(load_data())  # the sample every research question uses
print(f"{len(df)} respondents passed the attention check")

107 respondents passed the attention check


In [3]:
CUTOFF = 0.70  # the usual "acceptable" line for alpha


def cronbach_alpha(x: np.ndarray) -> float:
    """Alpha for an n-respondents x k-items array."""
    k = x.shape[1]
    item_var = x.var(axis=0, ddof=1).sum()
    total_var = x.sum(axis=1).var(ddof=1)
    return k / (k - 1) * (1 - item_var / total_var)


def bootstrap_ci(x: np.ndarray, rng: np.random.Generator) -> tuple[float, float]:
    boots = [cronbach_alpha(x[rng.integers(0, len(x), len(x))]) for _ in range(N_BOOT)]
    low, high = np.percentile(boots, [2.5, 97.5])
    return low, high


def item_table(x: np.ndarray, items: list[int]) -> pd.DataFrame:
    rows = []
    for j, item in enumerate(items):
        rest = np.delete(x, j, axis=1)
        rows.append(
            {
                "item": item,
                "item-total r": np.corrcoef(x[:, j], rest.sum(axis=1))[0, 1],
                "alpha if deleted": cronbach_alpha(rest),
            }
        )
    return pd.DataFrame(rows).round(3)


def report_reliability(df: pd.DataFrame) -> None:
    rng = np.random.default_rng(SEED)
    scales = {**{col: DASS_ITEMS[col] for col in DIMENSIONS}, "dass_total": range(1, 22)}
    for col, items in scales.items():
        x = df[[f"dass_{i}" for i in items]].dropna().to_numpy(dtype=float)
        alpha = cronbach_alpha(x)
        low, high = bootstrap_ci(x, rng)
        name = DIMENSIONS.get(col, "Total (21 items)")
        verdict = "acceptable" if alpha >= CUTOFF else "below 0.70"
        print(
            f"\n{name:<17} alpha = {alpha:.3f}  [95% CI {low:.3f}, {high:.3f}]  "
            f"n = {len(x)}, k = {x.shape[1]} -> {verdict}"
        )
        if col in DASS_ITEMS:
            display(item_table(x, DASS_ITEMS[col]).set_index("item"))

    print("\ncorrelation between the dimension scores (Pearson):")
    display(df[list(DIMENSIONS)].corr().round(2))


report_reliability(df)


Depression        alpha = 0.844  [95% CI 0.790, 0.880]  n = 107, k = 7 -> acceptable


,item-total r,alpha if deleted
item,,
3,0.546,0.831
5,0.516,0.838
10,0.613,0.822
13,0.674,0.812
16,0.544,0.831
17,0.693,0.808
21,0.662,0.818



Anxiety           alpha = 0.737  [95% CI 0.627, 0.810]  n = 107, k = 7 -> acceptable


,item-total r,alpha if deleted
item,,
2,0.302,0.742
4,0.477,0.700
7,0.515,0.692
9,0.294,0.742
15,0.499,0.694
19,0.627,0.658
20,0.479,0.701



Stress            alpha = 0.771  [95% CI 0.690, 0.824]  n = 107, k = 7 -> acceptable


,item-total r,alpha if deleted
item,,
1,0.478,0.746
6,0.444,0.754
8,0.615,0.716
11,0.526,0.738
12,0.499,0.742
14,0.414,0.760
18,0.495,0.743



Total (21 items)  alpha = 0.894  [95% CI 0.857, 0.917]  n = 107, k = 21 -> acceptable

correlation between the dimension scores (Pearson):


,dass_depression,dass_anxiety,dass_stress
dass_depression,1.00,0.55,0.68
dass_anxiety,0.55,1.00,0.55
dass_stress,0.68,0.55,1.00


**อ่านผล** ทั้งสามมิติผ่านเกณฑ์ 0.70 (ซึมเศร้า 0.844 วิตกกังวล 0.737 เครียด 0.771 และรวม 21 ข้อ 0.894)
มิติความวิตกกังวลอ่อนที่สุด ข้อ 2 และข้อ 9 มี item-total r ราว 0.30 แต่การตัดออกเพิ่มแอลฟาได้เพียง 0.005 จึงคงไว้ครบทุกข้อ
คะแนนสามมิติสัมพันธ์กันเอง 0.55 ถึง 0.68 ซึ่งยังแยกจากกันได้ จึงวิเคราะห์แยกมิติตามแผน